Auxiliary script to strip, clean and rename superblockify gkpg exports to geojson.

In [ ]:
import os
import glob
import shutil
import csv
import geopandas as gpd

In [ ]:
exports_path_gpkg = "../../dataexports/latest/superblockify_gpkg/"
exports_path_geojson = "../../dataexports/latest/superblockify/"
cities_path = "../../cities/"
cityfilename = "cities.csv"

In [ ]:
with open(cities_path+'meta/'+cityfilename, mode='r') as infile:
    reader = csv.reader(infile, delimiter=";")
    header = next(reader)
    cities = {rows[1]: {header[0]: rows[0], header[2]: rows[2], header[3]: rows[3], header[4]: rows[4]}  for rows in reader}

## Read, clean, and write data

In [ ]:
for fp in glob.glob(exports_path_gpkg + "*.gpkg"):
    if os.path.isfile(fp):
        sb_export = {}
        sb_export["edges"] = gpd.read_file(fp, layer="edges")
        sb_export["blocks"] = gpd.read_file(fp, layer="ltns")
        sb_export["boundary"] = gpd.read_file(fp, layer="graph_meta")
        city_name = os.path.basename(fp).split("_")[0]
        cityid = cities[city_name]["cityid"]
        print(city_name)

        # Edges
        edges = sb_export["edges"][["classification", "rel_increase_comp", "geometry"]]
        edges.to_file(exports_path_geojson+cityid+"-superblockify-edges.geojson", driver="GeoJSON", RFC7946="YES")

        # Blocks
        blocks = sb_export["blocks"][["classification", "street_length_total", "street_segment_count", "population", "area", "population_density",  "geometry"]]
        for c in ["street_length_total", "street_segment_count", "population", "area"]:
            blocks[c] = blocks[c].astype(int)
        blocks.to_file(exports_path_geojson+cityid+"-superblockify-blocks.geojson", driver="GeoJSON", RFC7946="YES")

        # Boundary
        # Temporary hack to replace superblockify's with our city boundary
        # For cities that have only shape files like Copenhagen, this does not work!
        shutil.copyfile("../../cities/cityexport/boundaries/"+cityid+".geojson", exports_path_geojson+cityid+"-city_boundary.geojson")
        